# ⚡ AI-Powered Energy Consumption Forecasting
## Exploratory Data Analysis & Model Walkthrough
---

## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid')
print('All imports successful ✓')

## 2. Load Dataset

In [ ]:
from preprocess import load_and_clean, add_features, FEATURE_COLS, TARGET_COL

df_raw = load_and_clean('../data/energy.csv')
print(f'Shape: {df_raw.shape}')
print(f'Date range: {df_raw.index.min()} → {df_raw.index.max()}')
df_raw.head(10)

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Timeseries
df_raw['Energy_kWh'].resample('D').mean().plot(ax=axes[0,0], color='#2196F3')
axes[0,0].set_title('Daily Average Consumption')

# Distribution
axes[0,1].hist(df_raw['Energy_kWh'], bins=50, color='#4CAF50', edgecolor='white')
axes[0,1].set_title('Energy Distribution')

# Hourly pattern
df_feat = add_features(df_raw)
df_feat.groupby('hour')['Energy_kWh'].mean().plot(ax=axes[1,0], marker='o', color='#FF5722')
axes[1,0].set_title('Avg by Hour of Day')

# Weekly pattern
days = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
weekly = df_feat.groupby('day_of_week')['Energy_kWh'].mean()
axes[1,1].bar(days, weekly.values, color=['#2196F3']*5 + ['#FF9800']*2)
axes[1,1].set_title('Avg by Day of Week')

plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
df_feat = add_features(df_raw)
print('Features:', FEATURE_COLS)
print(f'Shape after features: {df_feat.shape}')
df_feat[FEATURE_COLS + [TARGET_COL]].describe()

## 5. Train Gradient Boosting Model

In [ ]:
X = df_feat[FEATURE_COLS].values
y = df_feat[TARGET_COL].values

# Chronological split — no shuffling for time series!
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                   max_depth=5, random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)

mae  = mean_absolute_error(y_test, preds)
r2   = r2_score(y_test, preds)
print(f'MAE : {mae:.4f} kWh')
print(f'R²  : {r2:.4f}')

## 6. Actual vs Predicted

In [ ]:
test_index = df_feat.index[split:]
subset = slice(-14*24, None)  # last 14 days

plt.figure(figsize=(16, 5))
plt.plot(test_index[subset], y_test[subset], label='Actual', color='#1565C0', lw=2)
plt.plot(test_index[subset], preds[subset],  label='Predicted (GB)', color='#E53935', lw=1.8, ls='--')
plt.title('Actual vs Predicted — Last 14 Days of Test Set', fontweight='bold')
plt.ylabel('kWh')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

plt.figure(figsize=(9, 6))
importance.plot(kind='barh', color='#1976D2')
plt.title('Feature Importance (Gradient Boosting)', fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()